In [ ]:
import sys
import os

# Go to the cloned repo folder
repo_path = '/content/NLP-sequence-classification'

# Add to Python path
sys.path.append(repo_path)

# Change working directory to repo
os.chdir(repo_path)

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.besstie_data_loader import get_BESSTIE_splits
import pandas as pd
from src.tfidf_feature_extraction import tfidf_features, load_tfidf_features
from models.baseline.logistic_regression import LogisticRegressionModel

In [ ]:
#loading and getting the splits from the dataset
df_all, df_train, df_validation,df_test = get_BESSTIE_splits()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/711k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/70.6k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/415k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3747 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/313 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2183 [00:00<?, ? examples/s]

In [ ]:
# Extract TFIDF Features
X_train, X_validation, X_test, vectorizer = tfidf_features(
    df_train, df_validation, df_test,
    text_column='text',
    max_features=15000,
    save_path="./models/tfidf"
)

In [ ]:
#train the model
model = LogisticRegressionModel()
model.train_logistic_regression(X_train, df_train)

In [ ]:
#Evaluate the model
lr_eval_results = model.evaluate_logistic_regression(X_validation, df_validation)
print("\n📌 SARCASM DETECTION:")
print(f"   Accuracy:  {lr_eval_results['Sarcasm']['accuracy']:.4f}")
print(f"   Precision: {lr_eval_results['Sarcasm']['precision']:.4f}")
print(f"   Recall:    {lr_eval_results['Sarcasm']['recall']:.4f}")
print(f"   F1-Score:  {lr_eval_results['Sarcasm']['f1']:.4f}")

print("\n📌 SENTIMENT ANALYSIS:")
print(f"   Accuracy:  {lr_eval_results['Sentiment']['accuracy']:.4f}")
print(f"   Precision: {lr_eval_results['Sentiment']['precision']:.4f}")
print(f"   Recall:    {lr_eval_results['Sentiment']['recall']:.4f}")
print(f"   F1-Score:  {lr_eval_results['Sentiment']['f1']:.4f}")



📌 SARCASM DETECTION:
   Accuracy:  0.7827
   Precision: 0.3378
   Recall:    0.5682
   F1-Score:  0.4237

📌 SENTIMENT ANALYSIS:
   Accuracy:  0.8435
   Precision: 0.8333
   Recall:    0.8497
   F1-Score:  0.8414


model has hit the performance ceiling for TF-IDF + Logistic Regression on this dataset. The issue is not the features or parameters - it's the model capacity

Baseline result
"The TF-IDF + Logistic Regression baseline achieves 44.4% F1 for sarcasm detection and 83.8% F1 for sentiment analysis. This establishes a lower bound for comparison with transformer-based models."


Sarcasm --- 	Requires understanding context, not just keywords
Dataset Only 14% sarcastic overall - huge class imbalance
en-IN/en-UK have 7% sarcastic ---
	Very few examples to learn from
TF-IDF limitations	Cannot capture negation, irony, or pragmatic meaning


In [ ]:
# Save model
model.save_model("./models/baseline_logreg_yusrah_omar.pkl")

In [ ]:
# FINAL TEST EVALUATION
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
test_predictions = model.prediction_logistic_regression(X_test)

sarcasm_true_labels = df_test['Sarcasm'].astype(int).values
sentiment_true_labels = df_test['Sentiment'].astype(int).values

sarcasm_f1 = f1_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_precision = precision_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_recall = recall_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_accuracy = accuracy_score(sarcasm_true_labels, test_predictions['Sarcasm'])

sentiment_f1 = f1_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_precision = precision_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_recall = recall_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_accuracy = accuracy_score(sentiment_true_labels, test_predictions['Sentiment'])


print("FINAL TEST RESULTS")

print("\n📌 SARCASM DETECTION:")
print(f"   Accuracy:  {sarcasm_accuracy:.4f}")
print(f"   Precision: {sarcasm_precision:.4f}")
print(f"   Recall:    {sarcasm_recall:.4f}")
print(f"   F1-Score:  {sarcasm_f1:.4f}")

print("\n📌 SENTIMENT ANALYSIS:")
print(f"   Accuracy:  {sentiment_accuracy:.4f}")
print(f"   Precision: {sentiment_precision:.4f}")
print(f"   Recall:    {sentiment_recall:.4f}")
print(f"   F1-Score:  {sentiment_f1:.4f}")

FINAL TEST RESULTS

📌 SARCASM DETECTION:
   Accuracy:  0.7700
   Precision: 0.3179
   Recall:    0.5639
   F1-Score:  0.4066

📌 SENTIMENT ANALYSIS:
   Accuracy:  0.8191
   Precision: 0.8235
   Recall:    0.8011
   F1-Score:  0.8122
